<a href="https://colab.research.google.com/github/AlHartMos/IEU_courses/blob/main/principals_of_programming/PP_fundamentals_functions_contracts_evidence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Functions as Contracts + Evidence (Python)

**Core idea:** in 2026, writing code is cheap. **Trusting code is expensive.**  
So we treat every function as:

- a **contract** (specification: what it promises)
- plus **evidence** (tests + reasoning: why we believe it)

This notebook focuses on *how to design, document, and validate functions*.


## How we’ll use AI in this course

### Mode A — AI-allowed (default)
You may use AI to:
- draft code from a spec
- propose test cases / edge cases
- refactor for readability
- explain error messages

### Mode B — AI-free (short checks)
Sometimes you will do short “micro-checks” without AI:
- predict outputs
- spot contract violations
- reason about edge cases

**Rule:** If you can’t explain it, you can’t submit it.


## Deliverable for every function you write

For each function, you should be able to provide:

1. **Contract** (in English): inputs, outputs, edge cases, errors, side effects
2. **Docstring**: the contract in code
3. **Type hints**: a readable interface
4. **Evidence**: tests (and optionally a short “why I trust this” note)


---

# 1) The Contract Card

Before writing code, fill a tiny “contract card”.


### Contract Card (template)

- **Name:**  
- **Purpose (1 sentence):**  
- **Inputs:** types + meaning  
- **Output:** type + meaning  
- **Edge cases:** (empty, 0, negative, ties, NaN, weird strings, etc.)  
- **Errors:** what exceptions should be raised, when  
- **Side effects:** does it mutate inputs? print? write files?  
- **Complexity:** rough big-O (optional)


### Example contract card: `clamp`

We want a function that keeps a number inside an interval.

- **Name:** `clamp`
- **Purpose:** return `x` forced into `[lo, hi]`
- **Inputs:** `x` (float), `lo` (float), `hi` (float)
- **Output:** float in `[lo, hi]`
- **Edge cases:** `lo == hi`, `x` already inside interval
- **Errors:** raise if `lo > hi`
- **Side effects:** none (pure)


In [ ]:
from __future__ import annotations

def clamp(x: float, lo: float, hi: float) -> float:
    """Return x forced into the interval [lo, hi].

    Args:
        x: Value to clamp.
        lo: Lower bound (must be <= hi).
        hi: Upper bound.

    Returns:
        A value in [lo, hi].

    Raises:
        ValueError: If lo > hi.
    """
    if lo > hi:
        raise ValueError(f"Invalid interval: lo={lo} > hi={hi}")
    if x < lo:
        return lo
    if x > hi:
        return hi
    return x

# Quick evidence
assert clamp(3.0, 0.0, 10.0) == 3.0
assert clamp(-1.0, 0.0, 10.0) == 0.0
assert clamp(99.0, 0.0, 10.0) == 10.0


### Micro-check (AI-free)

Without running code, answer:

1. What should `clamp(5, 7, 7)` return?  
2. Should `clamp(5, 10, 0)` return something or raise an error? Why?


---

# 2) Docstrings: the contract *inside* the code

A **docstring** is a string literal placed right after `def ...:`.  
It becomes documentation accessible with `help(function)`.

Docstrings are not decoration: they define **intent** and **edge cases**.


### What a good docstring includes

- One-line summary: *what it does*
- Parameters: meaning, expected types, constraints
- Return value: meaning + type
- Exceptions: what you raise and when
- Notes: side effects, complexity, examples

We’ll use a compact “Google-ish” style for readability.


In [ ]:
def greet(name: str) -> str:
    """Return a friendly greeting.

    Args:
        name: Person name (non-empty string).

    Returns:
        Greeting message.

    Raises:
        ValueError: If name is empty or only whitespace.
    """
    if name.strip() == "":
        raise ValueError("name must be a non-empty string")
    return f"Hello, {name}!"


### Exercise 1 — Write the docstring first (then implement)

Write a function `mean(nums)` that returns the arithmetic mean of a list of numbers.

Contract requirements:
- Input: list/sequence of numbers (ints or floats)
- Output: float
- Edge case: empty list should raise `ValueError`
- Side effects: none


In [ ]:
# YOUR TURN (write the docstring first, then the code)

from typing import Sequence

def mean(nums: Sequence[float]) -> float:
    """TODO: write docstring."""
    # TODO: implement
    raise NotImplementedError


✅ **Solution (one possible version)**  
(Compare this with yours: same contract? same edge-case behavior?)


In [ ]:
from typing import Sequence

def mean(nums: Sequence[float]) -> float:
    """Return the arithmetic mean of a sequence of numbers.

    Args:
        nums: Sequence of numbers (ints or floats). Must be non-empty.

    Returns:
        The arithmetic mean as a float.

    Raises:
        ValueError: If nums is empty.
    """
    if len(nums) == 0:
        raise ValueError("mean() requires at least one number")
    return sum(nums) / len(nums)

assert mean([2, 4, 6]) == 4.0


---

# 3) Type hints: the contract *at the interface*

Type hints help you (and tools) understand function usage:
- better autocompletion
- earlier bug detection
- clearer APIs

Python doesn’t enforce type hints at runtime by default — they are *optional* but extremely valuable in real projects.


### Common hint patterns

- `int`, `float`, `str`, `bool`
- `list[T]`, `tuple[T1, T2]`, `dict[K, V]`
- `Sequence[T]` when you only need indexing/len (works for list/tuple)
- `Optional[T]` for “T or None”
- `Callable[[A, B], R]` for functions passed as arguments


In [ ]:
from typing import Optional, Callable

def safe_div(a: float, b: float) -> Optional[float]:
    """Return a/b, or None if b == 0."""
    if b == 0:
        return None
    return a / b

def apply(f: Callable[[float], float], x: float) -> float:
    """Apply a function f to x."""
    return f(x)

assert safe_div(10, 2) == 5.0
assert safe_div(10, 0) is None
assert apply(abs, -3.5) == 3.5


### Micro-check (AI-free)

For each function, say whether the type hints match the behavior:

1. `safe_div(10, 0)` returns `None`. Is `Optional[float]` correct?  
2. If `apply` is used with `abs`, why does `Callable[[float], float]` still work?


---

# 4) Evidence: tests are experiments

A contract is a claim. **Tests are evidence**.

We’ll use simple `assert` tests (fast, readable).  
For each function, create:
- a few **normal** tests
- a few **edge-case** tests
- at least one **“break it”** test (a case you suspect might fail)


In [ ]:
from typing import Callable

def run_tests(test_functions: list[Callable[[], None]]) -> None:
    """Run a list of test functions; raise immediately on first failure."""
    for t in test_functions:
        t()
    print(f"✅ {len(test_functions)} tests passed.")


### Example: tests for `mean`

Notice how tests reflect the contract:
- correct numeric value
- supports ints/floats
- empty input raises a clear error


In [ ]:
def test_mean_basic():
    assert mean([1, 2, 3]) == 2.0

def test_mean_singleton():
    assert mean([10]) == 10.0

def test_mean_mixed_types():
    assert mean([1, 2.5]) == 1.75

def test_mean_empty_raises():
    try:
        mean([])
        assert False, "Expected ValueError for empty input"
    except ValueError:
        pass

run_tests([test_mean_basic, test_mean_singleton, test_mean_mixed_types, test_mean_empty_raises])


### Exercise 2 — Ask AI for tests, then *judge them*

1. Ask an AI: “Propose tests for `clamp(x, lo, hi)` including edge cases.”  
2. Paste the proposed tests here.  
3. Mark which tests are redundant and which are missing.

*(In the course, you must include your AI prompt + what you changed.)*


In [ ]:
# Paste AI-suggested tests here, then edit them.
# Leave a short comment: what did AI miss? what did you add/remove?

def test_clamp_inside():
    assert clamp(5, 0, 10) == 5

run_tests([test_clamp_inside])


---

# 5) Exceptions are part of the contract

Errors are not “bad”: they are a way to enforce correct usage.
A good exception:
- uses the right exception type (`ValueError`, `TypeError`, ...)
- includes a message that helps the user fix the input


### Example: `gcd(a, b)` (contract-focused)

Contract:
- returns a **non-negative** gcd
- supports negative inputs
- raises if both are zero (undefined)


In [ ]:
def gcd(a: int, b: int) -> int:
    """Return the greatest common divisor of a and b (non-negative).

    Uses Euclid's algorithm.

    Args:
        a: An integer.
        b: An integer.

    Returns:
        The greatest common divisor of a and b (>= 0).

    Raises:
        ValueError: If a == 0 and b == 0 (gcd undefined).
    """
    a = abs(a)
    b = abs(b)
    if a == 0 and b == 0:
        raise ValueError("gcd(0, 0) is undefined")

    while b != 0:
        a, b = b, a % b
    return a

assert gcd(54, 24) == 6
assert gcd(-54, 24) == 6
assert gcd(0, 5) == 5


### Micro-check (AI-free)

What should happen for:
- `gcd(0, 0)`
- `gcd(0, -7)`
- `gcd(-12, -18)`


---

# 6) Side effects and mutation must be declared

Two functions can compute “the same thing” but with different contracts:

- **Pure**: returns a new object, does not mutate inputs
- **In-place**: mutates inputs, returns `None` or the same object

You must make this explicit in name + docstring.


In [ ]:
from typing import List

def add_one_pure(nums: List[int]) -> List[int]:
    """Return a new list where each element is +1 (does not mutate input)."""
    return [x + 1 for x in nums]

def add_one_in_place(nums: List[int]) -> None:
    """Add 1 to each element in nums *in place* (mutates input)."""
    for i in range(len(nums)):
        nums[i] += 1

a = [1, 2, 3]
b = add_one_pure(a)
assert a == [1, 2, 3]
assert b == [2, 3, 4]

add_one_in_place(a)
assert a == [2, 3, 4]


### Exercise 3 — Contract decision

You need a function that “cleans” a list of numbers by removing negative values.

Decide: should it be pure or in-place?  
Write the contract (docstring + type hints) accordingly.

Then implement it.


In [ ]:
from typing import List

def remove_negatives(nums: List[int]) -> List[int]:
    """TODO: Decide and write the contract.

    Hint: If you choose in-place, return type should probably be None.
    """
    raise NotImplementedError


✅ **Solution (pure version)**  
(An in-place version is also valid if clearly documented.)


In [ ]:
from typing import List

def remove_negatives(nums: List[int]) -> List[int]:
    """Return a new list with only the non-negative integers from nums.

    Args:
        nums: List of integers.

    Returns:
        New list containing elements x where x >= 0.
    """
    return [x for x in nums if x >= 0]

assert remove_negatives([3, -1, 0, -5, 2]) == [3, 0, 2]


---

# 7) API design: make functions hard to misuse

A function is better when:
- its parameters are hard to mix up
- defaults are safe
- the “dangerous” options are explicit


### Keyword-only parameters (clarity)

If a function has many options, force them to be used by name.


In [ ]:
def format_score(score: float, *, decimals: int = 1, percent: bool = True) -> str:
    """Format a score as a string.

    Args:
        score: A number (e.g., 0.0 to 1.0 if percent=True).
        decimals: Number of decimals.
        percent: If True, multiply by 100 and append '%'.

    Returns:
        A formatted string.
    """
    x = score * 100 if percent else score
    return f"{x:.{decimals}f}" + ("%" if percent else "")

assert format_score(0.1234) == "12.3%"
assert format_score(12.34, percent=False, decimals=2) == "12.34"


### The mutable default argument pitfall (contract violation)

Defaults are evaluated **once**, at function definition time.


In [ ]:
def append_bad(x: int, items: list[int] = []):
    """BAD: items is shared across calls."""
    items.append(x)
    return items

# Uncomment to see the surprising behavior:
# print(append_bad(1))
# print(append_bad(2))  # [1, 2]  (shared!)


✅ Fix: use `None` as the default, then create a new list inside.


In [ ]:
from typing import Optional

def append_good(x: int, items: Optional[list[int]] = None) -> list[int]:
    """Append x to items and return the list.

    If items is None, a fresh list is created (no cross-call sharing).
    """
    if items is None:
        items = []
    items.append(x)
    return items

assert append_good(1) == [1]
assert append_good(2) == [2]


---

# 8) Mini-project — From contract to evidence

### Problem
Write `parse_int(text)`.

Contract:
- Input: a string `text`
- Output: an `int` if `text` represents an integer (with optional leading/trailing spaces and optional + / - sign)
- If invalid, raise `ValueError` with a helpful message
- No side effects

You may use `int(text)` internally, but your job is to:
- define the contract clearly
- produce strong evidence tests
- make error messages user-friendly


In [ ]:
def parse_int(text: str) -> int:
    """TODO: Write a contract (docstring) and implement."""
    raise NotImplementedError


✅ **Reference solution**


In [ ]:
def parse_int(text: str) -> int:
    """Parse an integer from text.

    Accepts optional leading/trailing whitespace and an optional sign (+ or -).

    Args:
        text: Input string.

    Returns:
        The parsed integer.

    Raises:
        TypeError: If text is not a string.
        ValueError: If text does not represent a valid integer.
    """
    if not isinstance(text, str):
        raise TypeError("parse_int expects a string")

    s = text.strip()
    if s == "":
        raise ValueError("Invalid integer: empty string")

    # Optional sign
    if s[0] in "+-":
        digits = s[1:]
        if digits == "" or not digits.isdigit():
            raise ValueError(f"Invalid integer: {text!r}")
    else:
        if not s.isdigit():
            raise ValueError(f"Invalid integer: {text!r}")

    return int(s)

def test_parse_int_ok():
    assert parse_int("42") == 42
    assert parse_int("   -7 ") == -7
    assert parse_int("+0") == 0

def test_parse_int_bad():
    bad = ["", "   ", "3.14", "12a", "+", "-", "+-2", "++2"]
    for t in bad:
        try:
            parse_int(t)
            assert False, f"Expected ValueError for {t!r}"
        except ValueError:
            pass

run_tests([test_parse_int_ok, test_parse_int_bad])


---

# 9) “Why I trust this” (short write-up template)

For submissions, include a short paragraph:

- **Spec:** what the function should do (1–3 sentences)
- **Edge cases:** list the tricky inputs you considered
- **Evidence:** what tests you wrote and what they cover
- **AI usage (if any):** prompt + what you accepted/rejected and why


## Appendix: Quick checklist for good functions

- [ ] Name reflects behavior (`parse_int`, `remove_negatives`)
- [ ] Single responsibility
- [ ] Docstring states edge cases + exceptions
- [ ] Type hints reflect actual behavior
- [ ] Consistent return type
- [ ] Side effects are explicit (or avoided)
- [ ] Tests cover normal + edge + “break it”
